# NCBI Assembly Manifest Generation (Phase 1)

Downloads the current NCBI assembly summary from FTP, compares it against a
previous snapshot, and produces:

- `transfer_manifest.txt` — assemblies to download in Phase 2
- `removed_manifest.txt` — assemblies to archive in Phase 3
- `diff_summary.json` — human-readable summary of changes

All filtering (prefix range, limit) is applied here so downstream phases
receive a final, pre-filtered manifest.

In [ ]:
"""Imports and S3 client initialisation."""

from __future__ import annotations

import json
from pathlib import Path

from cdm_data_loaders.ncbi_ftp.assembly import FTP_HOST
from cdm_data_loaders.ncbi_ftp.manifest import (
    AssemblyRecord,
    compute_diff,
    download_assembly_summary,
    filter_by_prefix_range,
    parse_assembly_summary,
    write_diff_summary,
    write_removed_manifest,
    write_transfer_manifest,
    write_updated_manifest,
)
from cdm_data_loaders.utils.s3 import get_s3_client, split_s3_path

In [ ]:
"""Configure parameters."""

# Which NCBI database to sync: "refseq" or "genbank"
DATABASE = "refseq"

# Accession prefix filtering (3-digit, inclusive). Set to None to skip.
PREFIX_FROM: str | None = None  # e.g. "000"
PREFIX_TO: str | None = None  # e.g. "003"

# Maximum number of new/updated assemblies to include (None = unlimited)
LIMIT: int | None = None

# S3 path to the previous assembly summary snapshot (set to None on first run)
PREVIOUS_SUMMARY_S3: str | None = None  # e.g. "s3://cdm-lake/tenant-general-warehouse/kbase/datasets/ncbi/metadata/assembly_summary_refseq_prev.txt"

# S3 path where the new snapshot will be uploaded after diffing
SNAPSHOT_UPLOAD_S3: str | None = None  # e.g. "s3://cdm-lake/tenant-general-warehouse/kbase/datasets/ncbi/metadata/assembly_summary_refseq_curr.txt"

# Local output directory for manifest files
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# FTP hostname (default is the standard NCBI FTP server)
FTP_HOSTNAME = FTP_HOST

print(f"Database: {DATABASE}")
print(f"Prefix range: {PREFIX_FROM} -> {PREFIX_TO}")
print(f"Limit: {LIMIT}")
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
"""Download current assembly summary from NCBI FTP."""

raw_summary = download_assembly_summary(database=DATABASE, ftp_host=FTP_HOSTNAME)
current = parse_assembly_summary(raw_summary)
print(f"Parsed {len(current)} assemblies from current {DATABASE} summary")

In [ ]:
"""Load previous summary from S3 (or start fresh)."""

previous: dict[str, AssemblyRecord] | None = None

if PREVIOUS_SUMMARY_S3:
    s3 = get_s3_client()
    bucket, key = split_s3_path(PREVIOUS_SUMMARY_S3)
    resp = s3.get_object(Bucket=bucket, Key=key)
    prev_text = resp["Body"].read().decode("utf-8")
    previous = parse_assembly_summary(prev_text)
    print(f"Loaded {len(previous)} assemblies from previous snapshot")
else:
    print("No previous snapshot — all current 'latest' assemblies will be marked as new")

In [ ]:
"""Compute diff and apply filters."""

# Filter current assemblies by prefix range
filtered = filter_by_prefix_range(current, prefix_from=PREFIX_FROM, prefix_to=PREFIX_TO)
print(f"After prefix filter: {len(filtered)} assemblies")

# Also filter previous if present
filtered_prev = filter_by_prefix_range(previous, prefix_from=PREFIX_FROM, prefix_to=PREFIX_TO) if previous else None

# Compute diff
diff = compute_diff(filtered, previous_assemblies=filtered_prev)

print(f"New:        {len(diff.new)}")
print(f"Updated:    {len(diff.updated)}")
print(f"Replaced:   {len(diff.replaced)}")
print(f"Suppressed: {len(diff.suppressed)}")
print(f"Total to transfer: {len(diff.new) + len(diff.updated)}")
print(f"Total to remove:   {len(diff.replaced) + len(diff.suppressed)}")

# Apply limit if set
if LIMIT is not None:
    original_new = len(diff.new)
    original_updated = len(diff.updated)
    combined = diff.new + diff.updated
    limited = combined[:LIMIT]
    diff.new = [a for a in diff.new if a in set(limited)]
    diff.updated = [a for a in diff.updated if a in set(limited)]
    print(f"After limit ({LIMIT}): {len(diff.new)} new, {len(diff.updated)} updated")
    print(f"  (was {original_new} new, {original_updated} updated)")

In [ ]:
"""Write manifest files and upload snapshot to S3."""

# Write transfer manifest
transfer_path = OUTPUT_DIR / "transfer_manifest.txt"
paths = write_transfer_manifest(diff, filtered, transfer_path, ftp_host=FTP_HOSTNAME)
print(f"Transfer manifest: {len(paths)} entries -> {transfer_path}")

# Write removed manifest
removed_path = OUTPUT_DIR / "removed_manifest.txt"
removed = write_removed_manifest(diff, removed_path)
print(f"Removed manifest: {len(removed)} entries -> {removed_path}")

# Write updated manifest (for Phase 3 pre-overwrite archiving)
updated_path = OUTPUT_DIR / "updated_manifest.txt"
updated = write_updated_manifest(diff, updated_path)
print(f"Updated manifest: {len(updated)} entries -> {updated_path}")

# Write diff summary
summary_path = OUTPUT_DIR / "diff_summary.json"
summary = write_diff_summary(diff, summary_path, DATABASE, PREFIX_FROM, PREFIX_TO)
print(f"Diff summary -> {summary_path}")
print(json.dumps(summary["counts"], indent=2))

# Upload new snapshot to S3 for future diffing
if SNAPSHOT_UPLOAD_S3:
    s3 = get_s3_client()
    bucket, key = split_s3_path(SNAPSHOT_UPLOAD_S3)
    s3.put_object(Bucket=bucket, Key=key, Body=raw_summary.encode("utf-8"))
    print(f"Uploaded new snapshot to {SNAPSHOT_UPLOAD_S3}")
else:
    print("Skipping S3 snapshot upload (SNAPSHOT_UPLOAD_S3 not set)")